# Python + PyTorch Barebones Workbook — 2026-07-16

This workbook makes you write more of the implementation yourself. It moves from ordinary Python into algorithms and then PyTorch.

The progression is:

1. Loops and explicit state
2. Dictionaries and frequency counting
3. Pure functions over nested lists
4. Classes and persistent state
5. Generators and batching
6. Competitive puzzle: sliding window
7. Competitive puzzle: stack parsing
8. Matrix multiplication from scratch
9. Autograd training without an optimizer
10. A complete PyTorch classifier

Each section explains the mental model, contrasts correct and tempting alternatives, and describes consequences. Exercise cells contain contracts and tests but no solutions.

In [ ]:
import math
import random
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

random.seed(42)
torch.manual_seed(42)

print("Python/PyTorch practice environment ready")
print("PyTorch:", torch.__version__)

## 1. Loops, state, and invariants

A loop repeatedly transforms program state. The important skill is identifying the **invariant**: what must be true after every iteration.

To calculate a running total:

~~~python
total = 0
for value in values:
    total += value
~~~

After processing the first k values, the invariant is: total equals the sum of those k values. Variables such as minimum, maximum, and positive_count can be maintained in the same pass.

### Compare: one pass vs. repeated passes

~~~python
# Several passes over the same data
total = sum(values)
smallest = min(values)
largest = max(values)

# One explicit pass
total = 0
smallest = values[0]
largest = values[0]
for value in values:
    total += value
    if value < smallest:
        smallest = value
    if value > largest:
        largest = value
~~~

Built-ins are clearer in normal application code. The explicit version is useful practice because it exposes state transitions and allows several statistics to be computed in one traversal.

### Compare: safe initialization vs. magic sentinel values

~~~python
smallest = values[0]       # valid whenever non-empty input is required
smallest = 0               # wrong when every value is positive
smallest = float("inf")    # works for comparable numeric values
~~~

Initializing minimum to zero silently fails for values such as [5, 8, 10]. A clear function contract should either reject empty input or define an explicit empty result.

In [ ]:
# EXERCISE 1 — implement all statistics in one loop
#
# Requirements:
# - Do not use sum(), min(), max(), statistics, NumPy, or torch.
# - Raise ValueError for an empty list.
# - Return exactly these keys:
#   count, total, minimum, maximum, mean, positive_count
#
# Write the full implementation.

def analyze_numbers(values: list[float]) -> dict[str, float | int]:
    ...  # YOUR CODE


result = analyze_numbers([4.0, -2.0, 7.0, 0.0, 1.0])

assert result == {
    "count": 5,
    "total": 10.0,
    "minimum": -2.0,
    "maximum": 7.0,
    "mean": 2.0,
    "positive_count": 3,
}

try:
    analyze_numbers([])
    raise AssertionError("Empty input should raise ValueError")
except ValueError:
    pass

result

## 2. Dictionaries and frequency counting

A dictionary maps unique keys to values. Frequency tables use each observed item as a key and its count as the value.

~~~python
counts: dict[str, int] = {}

for item in items:
    if item not in counts:
        counts[item] = 0
    counts[item] += 1
~~~

The shorter form counts[item] = counts.get(item, 0) + 1 expresses the same transition.

### Compare: scanning a list vs. dictionary lookup

A list-based approach might search all earlier distinct items for every new item. With n distinct inputs this can require roughly n² comparisons.

A dictionary lookup is expected O(1), so counting n items is expected O(n). The consequence becomes visible at scale: doubling input can make a quadratic method roughly four times slower, while a linear method is roughly twice as slow.

### Compare: reading a missing key

~~~python
counts[item]              # KeyError if item is absent
counts.get(item, 0)       # returns 0 without modifying the dictionary
counts.setdefault(item, 0)  # inserts item with 0 if absent
~~~

get is appropriate when you need a default value only for this expression. setdefault mutates the dictionary, which can be surprising if you only intended to read.

### Deterministic tie-breaking

When several items have the same highest frequency, a problem must define which one wins. Without a rule, results can depend on insertion order. This workbook chooses the lexicographically smallest string.

In [ ]:
# EXERCISE 2 — frequency table and deterministic winner
#
# Implement both functions without collections.Counter.
#
# most_frequent requirements:
# - Raise ValueError for an empty dictionary.
# - Return (item, count).
# - If counts tie, choose the lexicographically smallest item.
# - Do not sort the entire dictionary; find the result in one pass.

def build_frequency_table(items: list[str]) -> dict[str, int]:
    ...  # YOUR CODE


def most_frequent(counts: dict[str, int]) -> tuple[str, int]:
    ...  # YOUR CODE


words = ["pear", "apple", "pear", "banana", "apple", "kiwi"]
counts = build_frequency_table(words)

assert counts == {"pear": 2, "apple": 2, "banana": 1, "kiwi": 1}
assert most_frequent(counts) == ("apple", 2)

try:
    most_frequent({})
    raise AssertionError("Empty counts should raise ValueError")
except ValueError:
    pass

counts

## 3. Pure functions and nested lists

A pure function returns a result without modifying its inputs or hidden global state. Purity makes behavior easier to test and reuse.

A table represented as list[list[float]] has an implicit shape:

~~~text
number of rows    = len(rows)
number of columns = len(rows[0])
~~~

Every row must have the same number of columns. Unlike a tensor, a Python nested list can be ragged, so your code must validate this itself.

### Compare: aliasing vs. copying

~~~python
output = rows                 # alias: both names refer to the same outer list
output[0][0] = 999            # rows is now modified too

output = [row.copy() for row in rows]
output[0][0] = 999            # original rows remains unchanged
~~~

Assignment copies a reference, not the nested data. A shallow outer copy such as rows.copy() is also insufficient because inner row lists remain shared.

### Compare: keepdim-style structure in plain Python

For each column, you need one mean and one standard deviation. Keeping them as lists shaped conceptually like (1, columns) makes their semantic role clear:

~~~python
means = [mean_column_0, mean_column_1, ...]
standardized_value = (rows[row_index][column_index] - means[column_index]) / stds[column_index]
~~~

If a column is constant, its standard deviation is zero. Dividing by it raises an error or produces invalid numeric output. This workbook defines a constant standardized column as all zeros.

In [ ]:
# EXERCISE 3 — standardize a rectangular table using plain Python
#
# Requirements:
# - Do not use torch, NumPy, statistics, or zip(*rows).
# - Do not mutate rows.
# - Use population variance: divide squared deviations by number of rows.
# - Constant columns become all zeros.
# - Raise ValueError when rows is empty, a row is empty, or rows are ragged.
# - Return a new list of rows with the same shape.
#
# You will likely need multiple loops. Write helper functions if useful.

def standardize_table(rows: list[list[float]]) -> list[list[float]]:
    ...  # YOUR CODE


table = [
    [1.0, 10.0, 5.0],
    [2.0, 20.0, 5.0],
    [3.0, 30.0, 5.0],
]
original_snapshot = [row.copy() for row in table]
standardized = standardize_table(table)
z = torch.tensor(standardized)

assert table == original_snapshot, "The input was mutated"
assert z.shape == (3, 3)
assert torch.allclose(z.mean(dim=0), torch.zeros(3), atol=1e-6)
assert torch.allclose(z[:, :2].std(dim=0, unbiased=False), torch.ones(2), atol=1e-6)
assert torch.equal(z[:, 2], torch.zeros(3))

for invalid in ([], [[]], [[1.0, 2.0], [3.0]]):
    try:
        standardize_table(invalid)
        raise AssertionError(f"Should reject {invalid}")
    except ValueError:
        pass

standardized

## 4. Classes and persistent state

A class groups data with operations that maintain valid state. An instance method receives self, which refers to that particular object.

A running mean needs only two pieces of state:

~~~text
count = how many values have been observed
total = sum of observed values
mean  = total / count
~~~

It does not need to retain every historical value.

### Compare: local variables vs. instance attributes

~~~python
def update(self, value):
    total = self.total + value   # local variable disappears after the call

def update(self, value):
    self.total += value          # stored on this instance
~~~

Forgetting self. creates a temporary local value, so the object appears not to update.

### Compare: instance attributes vs. class attributes

~~~python
class Bad:
    values = []                  # one list shared by every instance

class Good:
    def __init__(self):
        self.values = []         # separate list per instance
~~~

Mutable class attributes are a classic source of state leaking between objects. Initialize per-object mutable state inside __init__.

### Property vs. method

A property makes a computed value readable like data: tracker.mean. It should not perform expensive work or surprising mutation. Before any observations, this workbook makes mean raise ValueError rather than returning a misleading zero.

In [ ]:
# EXERCISE 4 — implement a stateful RunningMean
#
# Required interface:
# - RunningMean() starts with count=0 and total=0.0.
# - update(value) incorporates one number and returns None.
# - mean is a read-only @property.
# - reading mean before any update raises ValueError.
# - reset() restores the initial state.
#
# Write the complete class.

class RunningMean:
    ...  # YOUR CODE


first = RunningMean()
second = RunningMean()

first.update(2.0)
first.update(4.0)
second.update(100.0)

assert first.count == 2
assert first.total == 6.0
assert first.mean == 3.0
assert second.count == 1
assert second.mean == 100.0

first.reset()
assert first.count == 0
assert first.total == 0.0

try:
    _ = first.mean
    raise AssertionError("Empty RunningMean.mean should raise ValueError")
except ValueError:
    pass

## 5. Generators and batching

A generator produces values lazily using yield. Execution pauses at yield and resumes when the next value is requested.

~~~python
def countdown(n):
    while n > 0:
        yield n
        n -= 1
~~~

Calling countdown(3) does not immediately build [3, 2, 1]. It returns an iterator that produces one value at a time.

### Compare: returning all batches vs. yielding batches

~~~python
def all_batches(items, size):
    batches = []
    # append every batch
    return batches

def lazy_batches(items, size):
    # construct one batch
    yield batch
~~~

Returning a list retains every batch in memory. Yielding retains only the current batch, which matters for large datasets or streams.

### Compare: yielding a mutable buffer vs. a copy

~~~python
batch = []
yield batch
batch.clear()
~~~

The caller may still hold the same list object, so clearing it can make the already-yielded batch appear empty. Either create a fresh list after each yield or yield batch.copy().

### drop_last consequence

With 10 items and batch size 4, normal batches have sizes [4, 4, 2]. drop_last=True produces [4, 4], deliberately discarding two items. This can enforce fixed shapes during training but is usually inappropriate for evaluation.

In [ ]:
# EXERCISE 5 — implement a lazy batch generator
#
# Requirements:
# - Yield lists in original order.
# - Validate that batch_size is positive.
# - The input may be any iterable, not only a list; do not use len() or slicing.
# - If drop_last=False, yield the final smaller batch.
# - If drop_last=True, discard it.
# - Do not build all batches before yielding.

from collections.abc import Iterable, Iterator


def batch_iterable(
    items: Iterable[int],
    batch_size: int,
    drop_last: bool = False,
) -> Iterator[list[int]]:
    ...  # YOUR CODE


assert list(batch_iterable(range(10), 4)) == [
    [0, 1, 2, 3],
    [4, 5, 6, 7],
    [8, 9],
]
assert list(batch_iterable(range(10), 4, drop_last=True)) == [
    [0, 1, 2, 3],
    [4, 5, 6, 7],
]
assert list(batch_iterable((x * x for x in range(5)), 2)) == [
    [0, 1],
    [4, 9],
    [16],
]

try:
    list(batch_iterable(range(3), 0))
    raise AssertionError("batch_size=0 should raise ValueError")
except ValueError:
    pass

## 6. Competitive puzzle 1 — sliding window

### Problem pattern

Many array problems ask for the longest or shortest **contiguous** region satisfying a rule. Testing every start/end pair creates O(n²) candidate windows.

A sliding window keeps two indices:

~~~text
window = values[left:right + 1]
~~~

The right index expands the window. When the rule becomes invalid, the left index advances until validity is restored.

### Invariant

After the shrinking loop finishes:

- the current window contains at most k negative values;
- every element before left has been removed from the window state;
- right is included in the current window.

Because left and right each move only forward at most n times, the method is O(n), not O(n²).

### Compare: nested loops vs. sliding window

~~~text
Nested loops:
  for every start:
      for every end:
          recount negatives
Worst case: O(n³) if recounting each window

Sliding window:
  right visits each element once
  left removes each element at most once
Total: O(n)
~~~

### Tie-breaking

If two valid windows have the same maximum length, return the earliest one. Update the best answer only when current_length is strictly greater, not greater-or-equal.

In [ ]:
# COMPETITIVE PUZZLE 1 — longest window with at most k negatives
#
# Return (length, start_index, end_index_exclusive).
#
# Requirements:
# - The chosen contiguous window contains at most k values < 0.
# - Ties choose the earliest start.
# - Empty input returns (0, 0, 0).
# - Negative k raises ValueError.
# - Target O(n) time and O(1) extra space.
#
# Write the complete sliding-window implementation.

def longest_window_with_k_negatives(
    values: list[int],
    k: int,
) -> tuple[int, int, int]:
    ...  # YOUR CODE


assert longest_window_with_k_negatives(
    [1, -1, 2, -3, 4, 5, -2], 1
) == (4, 2, 6)  # [2, -3, 4, 5]

assert longest_window_with_k_negatives(
    [1, -1, 2, -3, 4, 5, -2], 0
) == (2, 4, 6)  # [4, 5]

assert longest_window_with_k_negatives([], 3) == (0, 0, 0)
assert longest_window_with_k_negatives([1, 2, 3], 0) == (3, 0, 3)

try:
    longest_window_with_k_negatives([1, 2], -1)
    raise AssertionError("Negative k should raise ValueError")
except ValueError:
    pass

## 7. Competitive puzzle 2 — stack parsing

A stack is last-in, first-out. Python lists implement the needed operations:

~~~python
stack.append(value)  # push
value = stack.pop()  # remove most recent
stack[-1]            # inspect most recent without removing
~~~

Opening brackets are pushed. A closing bracket must match the most recently opened bracket.

### Compare: a counter vs. a stack

A counter can verify that opening and closing counts match, but cannot verify nesting:

~~~text
([)] has:
one ( and one )
one [ and one ]
counts look balanced, but nesting is invalid
~~~

The stack remembers bracket type and order. When reading ), the most recent unmatched opener is [, revealing the mismatch immediately.

### Maximum depth

Stack length after pushing an opener equals current nesting depth. Track the largest observed length. Even if the string later becomes invalid, report the maximum depth reached before failure.

### Complexity

Each bracket is pushed and popped at most once: O(n) time. The stack can contain O(n) opening brackets in the worst case.

In [ ]:
# COMPETITIVE PUZZLE 2 — bracket validity and maximum depth
#
# Input contains only ()[]{} characters.
# Return (is_valid, maximum_depth_reached).
#
# Requirements:
# - A closer must match the most recent unmatched opener.
# - Leftover openers make the sequence invalid.
# - Empty input is valid with depth 0.
# - Do not repeatedly search or remove from the front of a list.
#
# Write the full parser.

def bracket_report(sequence: str) -> tuple[bool, int]:
    ...  # YOUR CODE


assert bracket_report("") == (True, 0)
assert bracket_report("([]{})") == (True, 2)
assert bracket_report("([{}])") == (True, 3)
assert bracket_report("([)]") == (False, 2)
assert bracket_report("((") == (False, 2)
assert bracket_report(")(") == (False, 0)

## 8. Matrix multiplication from scratch

For A shaped (m, k) and B shaped (k, n), the output C has shape (m, n).

Each output value is a dot product:

~~~text
C[i][j] = sum over p of A[i][p] * B[p][j]
~~~

That naturally becomes three loops: output row i, output column j, and shared dimension p.

### Compare: elementwise multiplication vs. matrix multiplication

~~~text
Elementwise:
A[i][j] * B[i][j]
Requires matching shapes and preserves positions.

Matrix multiplication:
sum(A[i][p] * B[p][j] for p)
Combines features and contracts the shared dimension.
~~~

### Shape validation

Before calculating:

- both matrices must be non-empty;
- every row within a matrix must have equal length;
- number of A columns must equal number of B rows.

Without validation, ragged inputs can fail halfway through with an IndexError, far from the actual cause. A clean ValueError at the boundary makes the contract obvious.

### Complexity

The direct algorithm performs m × n × k multiplications: O(mnk). PyTorch uses optimized native kernels, parallel hardware, and specialized memory layouts, but the mathematical operation is the same.

In [ ]:
# EXERCISE 8 — matrix multiplication using plain Python
#
# Requirements:
# - Use explicit loops; do not use torch, NumPy, zip(*matrix), or external libraries.
# - Validate non-empty, rectangular matrices.
# - Raise ValueError for incompatible shapes.
# - Do not mutate either input.
#
# Write any validation helpers you want.

def matmul_python(
    a: list[list[float]],
    b: list[list[float]],
) -> list[list[float]]:
    ...  # YOUR CODE


a = [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
b = [[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]]
result = matmul_python(a, b)

assert result == [[58.0, 64.0], [139.0, 154.0]]

torch_result = torch.tensor(a) @ torch.tensor(b)
assert torch.allclose(torch.tensor(result), torch_result)

for bad_a, bad_b in (
    ([], [[1.0]]),
    ([[1.0, 2.0], [3.0]], [[1.0], [2.0]]),
    ([[1.0, 2.0]], [[1.0, 2.0]]),
):
    try:
        matmul_python(bad_a, bad_b)
        raise AssertionError("Invalid matrices should raise ValueError")
    except ValueError:
        pass

result

## 9. Barebones PyTorch training with autograd

A one-feature linear model is:

~~~text
prediction = x * weight + bias
~~~

Training repeatedly measures error, calculates gradients, and updates parameters.

~~~python
prediction = x * weight + bias
loss = ((prediction - target) ** 2).mean()
loss.backward()

with torch.no_grad():
    weight -= learning_rate * weight.grad
    bias -= learning_rate * bias.grad

weight.grad = None
bias.grad = None
~~~

### Why no_grad surrounds the update

weight and bias require gradients and are leaf tensors. Updating them normally with an in-place operation would itself become part of the graph or raise an error about modifying a leaf tensor. Parameter updates are algorithmic bookkeeping, not differentiable model computation, so they belong inside no_grad.

### Compare: clearing vs. accumulating gradients

~~~python
# Clear after every update
weight.grad = None

# Do not clear
# Next backward adds a new gradient to the previous one
~~~

Accumulation is intentional for micro-batch training but wrong for ordinary independent steps. Updates become mixtures of current and stale gradients.

### Compare: tensor loss vs. loss.item()

The tensor loss retains its computation graph and supports backward. loss.item() returns a detached Python float for logging. Calling backward on the float is impossible.

### History as a debugging tool

Store float(loss.item()) each step. A generally falling history confirms learning. A constant loss suggests missing updates; exploding or NaN loss suggests an excessive learning rate, invalid data, or unstable operations.

In [ ]:
# EXERCISE 9 — train y = 4x - 2 without nn.Module or an optimizer
#
# Requirements:
# - Create scalar weight and bias tensors with requires_grad=True.
# - Use mean squared error.
# - Implement forward, backward, parameter updates, and gradient clearing.
# - Append a Python float loss to history every step.
# - Return detached copies: (weight, bias, history).
# - Do not use nn.Linear, nn.Module, torch.optim, or analytic least squares.
#
# Write the complete training function.

def train_linear_regression(
    x: torch.Tensor,
    target: torch.Tensor,
    steps: int,
    learning_rate: float,
) -> tuple[torch.Tensor, torch.Tensor, list[float]]:
    ...  # YOUR CODE


x = torch.linspace(-3, 3, 61)
target = 4 * x - 2

weight, bias, history = train_linear_regression(
    x,
    target,
    steps=300,
    learning_rate=0.05,
)

assert weight.requires_grad is False
assert bias.requires_grad is False
assert len(history) == 300
assert history[-1] < history[0]
assert abs(weight.item() - 4.0) < 0.05
assert abs(bias.item() + 2.0) < 0.05

weight.item(), bias.item(), history[-1]

## 10. Full PyTorch classifier

PyTorch abstractions remove repeated plumbing without changing the training logic:

- nn.Module registers parameters and defines forward computation.
- TensorDataset pairs aligned tensors.
- DataLoader creates shuffled mini-batches.
- CrossEntropyLoss converts logits and class indices into scalar loss.
- An optimizer updates registered parameters.

### Shape contract

~~~text
features: (batch, 2), float32
labels:   (batch,), int64 class indices
logits:   (batch, 2), raw floating scores
~~~

CrossEntropyLoss expects raw logits. Applying softmax first still runs, but changes the objective and compresses useful score differences.

### Compare: train and evaluation modes

~~~python
model.train()
# dropout and batch normalization use training behavior

model.eval()
with torch.no_grad():
    # deterministic evaluation behavior, no graph allocation
~~~

eval() does not disable gradients. no_grad() does not switch dropout or batch normalization. Use both for evaluation.

### Compare: averaging batch accuracies vs. counting examples

If the final batch is small, giving its percentage equal weight distorts the dataset metric. Accumulate number correct and number seen, then divide once.

### Why this exercise uses a linear boundary

Labels are generated from x0 + 0.75*x1 > 0. A single linear layer can represent that rule. If it fails, the issue is training code rather than insufficient model capacity.

In [ ]:
# EXERCISE 10 — implement a complete binary classifier pipeline
#
# Implement BinaryClassifier and train_classifier.
#
# BinaryClassifier requirements:
# - Subclass nn.Module.
# - Contain one nn.Linear(2, 2).
# - forward returns raw logits.
#
# train_classifier requirements:
# - Build TensorDataset and shuffled DataLoader with batch_size=32.
# - Create the model, CrossEntropyLoss, and Adam optimizer with lr=0.05.
# - For every batch: zero gradients, forward, loss, backward, step.
# - Record the correctly example-weighted mean epoch loss.
# - Train for exactly the requested number of epochs.
# - Return (model, history).
#
# Write substantially all code yourself.

class BinaryClassifier(nn.Module):
    ...  # YOUR CODE


def train_classifier(
    features: torch.Tensor,
    labels: torch.Tensor,
    epochs: int,
) -> tuple[nn.Module, list[float]]:
    ...  # YOUR CODE


torch.manual_seed(123)
features = torch.randn(400, 2)
labels = (features[:, 0] + 0.75 * features[:, 1] > 0).long()

model, history = train_classifier(features, labels, epochs=30)

model.eval()
with torch.no_grad():
    logits = model(features)
    predictions = logits.argmax(dim=1)
    final_accuracy = (predictions == labels).float().mean().item()

assert logits.shape == (400, 2)
assert len(history) == 30
assert history[-1] < history[0]
assert final_accuracy >= 0.95, f"Only reached {final_accuracy:.1%}"

final_accuracy, history[-1]

## After completing the workbook

Refactor only after the tests pass.

Suggested review:

- Identify the invariant in every loop.
- State the time and space complexity of both competitive puzzles.
- Mark which functions mutate state and which are pure.
- Compare your manual batch generator with DataLoader.
- Compare manual gradient updates with an optimizer.
- Add one deliberately broken line, predict the symptom, and verify it.
- Rewrite one long function into two or three helpers with clear contracts.

Optional extensions:

- Modify the sliding-window puzzle to allow at most k odd values.
- Make bracket_report reject non-bracket characters.
- Add matrix addition and transpose to the plain-Python matrix utilities.
- Add a train/validation split to the classifier.
- Save and reload the classifier state_dict.
- Add per-class accuracy instead of only overall accuracy.